# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook guides you in loading, examining, and analyzing the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

- **Title**: Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells
- **URL**: https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset from the Croissant schema and inspect the main metadata and available record sets.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset via mlcroissant
dataset = mlc.Dataset(croissant_url)
# Show main metadata
ds_meta = dataset.metadata
print(f"Dataset title: {ds_meta.name}")
print(f"Published: {ds_meta.datePublished}\n")
print(f"Description: {ds_meta.description}\n")
print(f"Keywords: {getattr(ds_meta, 'keywords', None)}")
print(f"Record sets: {len(ds_meta.recordSet) if hasattr(ds_meta, 'recordSet') else 0}")

## 2. Data Overview
List available record sets and their fields, referencing everything by `@id` as defined in the Croissant schema.

In [ ]:
# List all record sets and their fields by @id
from pprint import pprint

record_sets = ds_meta.recordSet if hasattr(ds_meta, 'recordSet') else []

overview = []
for record_set in record_sets:
    rsid = getattr(record_set, '@id', None)
    rsname = getattr(record_set, 'name', None)
    fields = getattr(record_set, 'field', [])
    # Ensure fields is always a list
    if not isinstance(fields, list):
        fields = [fields]
    field_overview = []
    for f in fields:
        fid = getattr(f, '@id', None)
        fname = getattr(f, 'name', None)
        ftype = getattr(f, 'dataType', None)
        field_overview.append(f"{fname} (@id: {fid}, type: {ftype})")
    overview.append({'record_set_id': rsid, 'record_set_name': rsname, 'fields': field_overview})

print("Record sets and fields in this dataset:")
pprint(overview)

## 3. Data Extraction
Load records from each record set into a separate DataFrame using their `@id`. Example shown for the first record set.

> Note: Every `record_set`, `field`, and column should be referenced by its `@id`, and you should adapt to which sets and fields are actually present.

In [ ]:
dataframes = {}

record_sets = ds_meta.recordSet if hasattr(ds_meta, 'recordSet') else []
record_set_ids = [getattr(rs, '@id', None) for rs in record_sets]
print(f"Record set IDs found: {record_set_ids}")
# For demonstration, load only the first record set (if available)
if record_set_ids:
    rec_id = record_set_ids[0]
    recs = list(dataset.records(record_set=rec_id))
    df = pd.DataFrame(recs)
    dataframes[rec_id] = df
    print(f"Loaded DataFrame for {rec_id} with {len(df)} records.")
    print(f"Columns: {df.columns.tolist()}")
    df.head()
else:
    print("No record sets found in the dataset.")

## 4. Exploratory Data Analysis (EDA)
Apply basic data processing steps such as filtering, normalization, and grouping by categorical fields.

*All references to fields below use their `@id`, as listed in previous sections.*

In [ ]:
# --- Replace the following IDs with ones from your dataset overview ---
if dataframes and record_set_ids:
    # Select a field to use for numeric analysis. Replace with appropriate @id.
    record_set_id = record_set_ids[0]
    df = dataframes[record_set_id]
    # Try to auto-detect a numeric field (use the first float/integer column we find)
    numeric_field = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
    if numeric_field is None:
        print("No numeric fields found for EDA.")
    else:
        threshold = df[numeric_field].mean()
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f} (mean):")
        print(filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to auto-detect a groupable field (the first 'object' dtype field with low cardinality)
        group_field = None
        for col in df.columns:
            if pd.api.types.is_object_dtype(df[col]) and df[col].nunique() < 10:
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"\nGrouped data by {group_field} (mean {numeric_field}):")
            print(grouped_df.head())
        else:
            print("No suitable categorical group field found.")
else:
    print("No record sets loaded for EDA.")

## 5. Visualization
Visualize distributions of numeric fields (or categorical breakdowns if suitable fields exist).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
if dataframes and record_set_ids and numeric_field is not None:
    df = dataframes[record_set_id]
    # Plot histogram of numeric_field
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field].dropna(), bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    if group_field:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion

In this notebook, you loaded a complex FAIR² dataset using `mlcroissant`, explored its record sets and fields by their unique `@id`, extracted tabular data, performed basic filtering and normalization, and visualized key distributions.

This workflow can be extended to all record sets and fields, or adapted for custom downstream scientific analysis.